In [1]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
sys.path.insert(1, '../')

In [2]:
import pyFBS
from pyFBS.utility import *
import pickle
import pyvista as pv

In [3]:
import numpy as np 
import pyvista as pv
from numpy import cross, eye
from scipy.linalg import expm, norm
import sys

In [4]:
import pyvista as pv
import numpy as np
import numpy as np

import cv2

In [5]:
import warnings
warnings.filterwarnings("ignore")

In [64]:
def M(axis, theta):
    """
    Euler-Rodrigues formula
    """
    t = expm(cross(eye(3), axis/norm(axis)*(theta)))
    #print(theta)
     
    return t

    
#Create a unit vector
def unit_vector(vec):
    unit = vec / np.linalg.norm(vec)
    return unit

points = np.array([[0.5,0.5,0.5],
                   [0,0.5,0.5],
                   [0.5,0,0.5],
                   [0.5,0.5,0]])

# Find the angle between two unit vector
def angle(vector1, vector2):
    """ Returns the angle in radians between given vectors"""
    v1_u = unit_vector(vector1)
    v2_u = unit_vector(vector2)
    minor = np.linalg.det(
        np.stack((v1_u[-2:], v2_u[-2:]))
    )
    if minor == 0:
        sign = 1
    else:
        sign = -np.sign(minor)
    dot_p = np.dot(v1_u, v2_u)
    dot_p = min(max(dot_p, -1.0), 1.0)
    return sign * np.arccos(dot_p)


def rotation_matrix_from_vectors(vec1, vec2):
    """ Find the rotation matrix that aligns vec1 to vec2
    :param vec1: A 3d "source" vector
    :param vec2: A 3d "destination" vector
    :return mat: A transform matrix (3x3) which when applied to vec1, aligns it with vec2.
    """
    vec1 += np.random.random(3)/1e15
    a, b = (vec1 / np.linalg.norm(vec1)).reshape(3), (vec2 / np.linalg.norm(vec2)).reshape(3)
    #print(a,b)

    if (np.abs(a) == np.abs(b)).all():
        return np.diag([1,1,1])
    else:
        v = np.cross(a, b)
        c = np.dot(a, b)
        s = np.linalg.norm(v)
        kmat = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        rotation_matrix = np.eye(3) + kmat + kmat.dot(kmat) * ((1 - c) / (s ** 2))

        return rotation_matrix

def unit_vector(vector):
    """ Returns the unit vector of the vector.  """
    return vector / np.linalg.norm(vector)

def angle_between(v1, v2):
    """ Returns the angle in radians between vectors 'v1' and 'v2'::

            >>> angle_between((1, 0, 0), (0, 1, 0))
            1.5707963267948966
            >>> angle_between((1, 0, 0), (1, 0, 0))
            0.0
            >>> angle_between((1, 0, 0), (-1, 0, 0))
            3.141592653589793
    """
    v1_u = unit_vector(v1)
    v2_u = unit_vector(v2)
    return np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0))
    
class Accelerometer():
    def __init__(self,p,N,mesh = None):

        # Creates an accelerometer
        self.box = pv.Box()
        self.box.translate([1,1,1])
        self.box.points /= 2

        p.add_mesh(self.box,opacity = 0.3, show_edges  = True, color = "#8c8c8c")

        ray_x = pv.Line([0,0,0], [1,0,0])
        p.add_mesh(ray_x, color="r", line_width=3)
        ray_y = pv.Line([0,0,0], [0,1,0])
        p.add_mesh(ray_y, color="g", line_width=3)
        ray_z = pv.Line([0,0,0], [0,0,1])
        p.add_mesh(ray_z, color="b", line_width=3)
        

        self.accelerometer = [self.box,ray_x,ray_y,ray_z]
        self.N = int(N*4)
    
        self.local_orientation = np.asarray([[1, 0, 0],
                          [0, 1, 0],
                          [0, 0, 1]])

        self.local_widgets = np.array([[0.0, 0.0, 0.0],
                   [-0.5, 0, 0],
                   [0, -0.5, 0],
                   [0, 0, -0.5]])
        
        self.local_normals = np.asarray([[1, 0, 0],
                                        [0, 1, 0],
                                        [0, 0, 1],
                                        [-1, 0, 0],
                                        [0, -1, 0],
                                        [0, 0, -1]]).T
        
        ray_size = 4
        self.local_rays = np.asarray([[1, 0, 0],
                                        [0, 1, 0],
                                        [0, 0, 1],
                                        [-1, 0, 0],
                                        [0, -1, 0],
                                        [0, 0, -1]]).T * ray_size

        self.mesh = mesh
        self.mesh.compute_normals(auto_orient_normals = True,inplace = True)

        self.def_rot = 0
        self.turn_on = False
        
        
    def translate(self,point,snap = False):
        
        point1x = point + self.local_rays[:,0]
        point2x = point + self.local_rays[:,3]
        
        point1y = point + self.local_rays[:,1]
        point2y = point + self.local_rays[:,4]
        
        point1z = point + self.local_rays[:,2]
        point2z = point + self.local_rays[:,5]
        
                
        points_x, ind_x = self.mesh.ray_trace(point1x,point2x)
        points_y, ind_y = self.mesh.ray_trace(point1y,point2y)
        points_z, ind_z = self.mesh.ray_trace(point1z,point2z)
        
        points = np.vstack([points_x,points_y,points_z])
        ind = np.hstack([ind_x,ind_y,ind_z])

        #print("X",ind_x.shape)
        #print("Y",ind_y.shape)
        #print("Z",ind_z.shape)
        #print("all",ind.shape,points.shape)
        
        rot = np.diag([1,1,1])
        
        if points.size != 0:        
            list_ind = []
            # go through all the intersections
            for i in range(len(points)):
                _point = points[i]
                p1 = _point
                p2 = point
                gg  = np.sqrt( ((p1[0]-p2[0])**2)+((p1[1]-p2[1])**2)+((p1[2]-p2[2])**2) )
                
                list_ind.append(gg)
            
            # find the closest to the acc center
            _sel = np.argmin(list_ind)
            
            
            #print("selected_index",_sel)
            
            # Find the nearest normal
            v2 = self.mesh.cell_normals[int(ind[_sel])]
            th = []
            for _loc in self.local_normals.T:
                th.append(angle_between(_loc, v2))
                
            
            
            closest_orient = self.local_normals.T[np.argmin(th)]
            
            #print("orient",closest_orient)
            #print(points,ind)
            
            
            # find orientation between acc orientation and cell normal and then push acc 0.5 away from the normal
            
            f = self.mesh.cell_normals[int(ind[_sel])] #+  np.random.random(3)/1e10
            t = closest_orient + np.random.random(3)/1e10  # just so that the math works
            
            rot = rotation_matrix_from_vectors(t,f)
            
            point = points[_sel] + f/2  #+ sum_plus

            #print(rot)
            #print(t,f,rot@f)
            
            # simply push in normal direction 
            
            #print(rot,"rot!")
                
        else:
            pass
            #print(points,ind)
            
        #print( points[_sel] + f/2)
        
        _new = point - self.box.center_of_mass() #+ [0.5, 0.5, 0.5]
        

        
        
        if snap:
            
            
            # do all the translation stuff!
            for item in self.accelerometer:
                item.translate(_new)  
                
            p.sphere_widgets[self.N+0].SetCenter(point)
            #for i in range(3):
                #p.sphere_widgets[self.N +1+ i].SetCenter(_new + rot@np.asarray(p.sphere_widgets[self.N  +1+ i].GetCenter()))
                #self.local_widgets[i+1, :] = rot@self.local_widgets[i+1, :]
            
            t_new = self.box.center_of_mass()

            for k in range(3):    
                self.local_widgets[k+1, :] = rot@self.local_widgets[k+1, :]
                p.sphere_widgets[self.N +  k+1].SetCenter(self.local_widgets[k+1, :]  + t_new)

            

            # orient the local csys of accelerometer with the new rotation
            self.local_orientation = (rot @ (self.local_orientation).T).T 
            self.local_normals = rot @ self.local_normals
            self.local_rays = rot @ self.local_rays
            
            
            #rotate everything within accelerometer
            for item in self.accelerometer:
                item.points = (rot @ (item.points - t_new).T).T + t_new 
                
                
            
                
        else:
            for item in self.accelerometer:
                item.translate(_new)
            for i in range(4):
                p.sphere_widgets[self.N  + i].SetCenter(_new + np.asarray(p.sphere_widgets[self.N + i].GetCenter()))
        
        
        
    def callback(self,point, i):
        print(i,point)
        # 3D translation in space
        if i == 0:
            
            self.translate(point,snap = True)

        else:
            if self.turn_on:
                # get the center of acc
                _new = self.box.center_of_mass()
                _vec1 = np.asarray(point-_new)
                _vec2 = (np.asarray(self.local_widgets[i,:])) 

                theta = angle(_vec1, _vec2)
                # define the rotational matrix based on angle of rotation
                rot = M(self.local_orientation[i-1, :], theta)

                # rotate everything within accelerometer
                for item in self.accelerometer:
                    item.points = (rot @ (item.points - _new).T).T + _new 

                # orient the local csys of accelerometer with the new rotation
                self.local_orientation = (rot @ (self.local_orientation).T).T 
                self.local_normals = rot @ self.local_normals
                self.local_rays = rot @ self.local_rays



                # position all widgets to the new position
                for k in range(4):
                    self.local_widgets[k, :] = rot@self.local_widgets[k, :]
                    p.sphere_widgets[self.N +  k].SetCenter(self.local_widgets[k, :]  + _new)

In [96]:
view3D = pyFBS.view3D(show_origin = False)

In [97]:
stl = pyFBS.example_lab_testbench["STL"]["A"]

mesh = pv.PolyData(stl)
mesh.points /= 10

view3D.plot.add_mesh(mesh,name = "ts",color = "#D3D3D3")
#view3D.plot.add_mesh(mesh,name = "ts_p",color = "b",style = "points")

(vtkRenderingOpenGL2Python.vtkOpenGLActor)000001D5B15D85E8

In [98]:
p = view3D.plot
#p.background_color = "#D4D4D4"

def add_accelerometer(point):
    try:
        i = int(len(view3D.plot.sphere_widgets)/4)
    except:
        i = 0
    _gg = Accelerometer(view3D.plot,i,mesh = mesh)
    view3D.plot.add_sphere_widget(_gg.callback, center=points, color = ["k","r","g","b"],radius = 0.1)
    
    _gg.turn_on = True
    
    _gg.translate(point)

In [99]:
view3D.plot.enable_point_picking(callback = add_accelerometer,color = "r",show_message="")
view3D.plot.add_text("Press P too add accelerometer", font_size = 12,color = "k")

(vtkRenderingAnnotationPython.vtkCornerAnnotation)000001D5B15D8E28

3 (0.5, 0.5, 0.0)
0 (5.661994674471566, 34.947235246333136, 0.7473514844344624)
0 (5.246960653411553, 35.59591556772048, 0.4811845697868704)
0 (5.217757641580466, 36.0727829015857, 0.8252484482642732)
3 (0.5, 0.5, 0.0)
0 (2.1857570043333325, 35.377573323239865, 0.8426463125731738)
